# 🏦 AI-Powered Customer Intelligence PoC
## Claude + Teradata ClearScape Analytics

This notebook walks through the full PoC in 5 steps:
1. **Setup** — Connect to Teradata via config.yaml
2. **Data Overview** — Understand the 360° customer dataset
3. **Progressive Segmentation** — RFM → +Risk → +Demographics
4. **Elbow & Silhouette** — Find optimal k at each phase
5. **Business Insight** — The story for stakeholders

---
> **Environment**: Teradata ClearScape Analytics  
> **AI Layer**: Claude Desktop + MCP (Model Context Protocol)  
> **ML Engine**: Teradata VAL tda_kmeans (in-database, no data movement)

## 0. Setup

In [ ]:
import yaml, pandas as pd, numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from teradataml import create_context, DataFrame, remove_context
from teradataml.analytics.mle import KMeans, KMeansPredict
from sklearn.metrics import silhouette_score

with open('../config.yaml') as f:
    cfg = yaml.safe_load(f)
td, exp, mcp = cfg['teradata'], cfg['experiment'], cfg.get('mcp', {})

create_context(host=td['host'], username=td['user'], password=td['password'], database=exp['database'])
print('Connected to Teradata ✅')

## 1. Data Overview

| Table | Rows | Description |
|---|---|---|
| `transactions` | 799,477 | 2 years of transaction history |
| `customer_demographics` | 50,000 | Age, income, employment |
| `credit_risk` | 50,000 | Credit score, loan exposure, late payments |
| `customer_features_rfm` | 50,000 | Aggregated RFM features |
| `segmentation_dataset` | 50,000 | Combined 360° feature set |

In [ ]:
seg = DataFrame.from_table(f"{exp['database']}.segmentation_dataset").to_pandas()
print(f'Shape: {seg.shape}')
seg.describe().round(2)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
fig.suptitle('Customer Feature Distributions', fontsize=15, fontweight='bold')
feats = ['monetary','frequency','recency_score','age','income','credit_score']
colors = ['#00d4ff','#ff6b6b','#00ff9d','#ffd700','#ff8c42','#c77dff']
for ax, f, c in zip(axes.flat, feats, colors):
    ax.hist(seg[f].dropna(), bins=40, color=c, alpha=0.8, edgecolor='white', linewidth=0.3)
    ax.set_title(f.replace('_',' ').title(), fontweight='bold')
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('../results/feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. The Pipeline

```
transactions (799K)
    │ SUM(amount), COUNT(*), days since last tx
    ▼
customer_features_rfm (50K)   ← Behavioral
    │ JOIN demographics + credit_risk
    ▼
segmentation_dataset (50K)    ← 360° view
    │ Min-Max scaling
    ▼
segmentation_*_scaled         ← KMeans-ready
    │ val.tda_kmeans IN-DATABASE
    ▼
Customer Segments             ← Actionable
```
**All computation stays inside Teradata.**

## 3. Progressive Segmentation Experiment

In [ ]:
def run_elbow(table, features, k_range=range(2,9)):
    td_df = DataFrame.from_table(f"{exp['database']}.{table}")
    recs  = []
    for k in k_range:
        print(f'  k={k}...', end=' ', flush=True)
        try:
            m   = KMeans(data=td_df, n_clusters=k, id_column='customer_id',
                         center_columns=features, max_iter=100, seed=42)
            mdf = m.result.to_pandas()
            wss = None
            for _, r in mdf[mdf.td_modelinfo_kmeans.notna()].iterrows():
                s = str(r.td_modelinfo_kmeans)
                if 'Total_WithinSS' in s:
                    try: wss = float(s.split(':')[-1])
                    except: pass
            p   = KMeansPredict(object=m, newdata=td_df, id_column='customer_id', accumulate=features)
            pdf = p.result.to_pandas().sample(min(8000, len(p.result.to_pandas())), random_state=42)
            sil = silhouette_score(pdf[features], pdf['td_clusterid_kmeans'])
            recs.append({'k':k,'within_ss':wss,'silhouette':round(sil,4)})
            print(f'✅ wss={wss:.1f} sil={sil:.3f}')
        except Exception as e:
            print(f'❌ {e}'); recs.append({'k':k,'within_ss':None,'silhouette':None})
    return pd.DataFrame(recs)

In [ ]:
print('▶ Phase 1: RFM Only')
r1 = run_elbow(exp['tables']['rfm'], exp['features']['rfm'])
print('\n▶ Phase 2: RFM + Credit Risk')
r2 = run_elbow(exp['tables']['risk'], exp['features']['risk'])
print('\n▶ Phase 3: Full 360° (RFM + Demo + Risk)')
r3 = run_elbow(exp['tables']['full'], exp['features']['full'])

## 4. Elbow & Silhouette Plots

In [ ]:
experiments = [
    ('RFM Only',          r1, '#00d4ff'),
    ('RFM + Risk',        r2, '#ff6b6b'),
    ('RFM + Demo + Risk', r3, '#00ff9d'),
]
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.patch.set_facecolor('#0f1117')
fig.suptitle('KMeans Experiments — Elbow & Silhouette\nTeradata ClearScape In-Database ML',
             color='white', fontsize=14, fontweight='bold')
for col, (label, df, color) in enumerate(experiments):
    v  = df.dropna()
    ks, wss, sil = v.k.values, v.within_ss.values, v.silhouette.values
    bk = ks[np.argmax(sil)]
    ax1 = axes[0, col]; ax1.set_facecolor('#1a1d27')
    ax1.plot(ks, wss, color=color, lw=2.5, marker='o', markersize=7, markerfacecolor='white')
    ax1.set_title(f'{label}\nElbow', color='white', fontweight='bold')
    ax1.set_xlabel('k', color='#aaa'); ax1.set_ylabel('Within-SS', color='#aaa')
    ax1.tick_params(colors='#aaa')
    for s in ax1.spines.values(): s.set_color('#333344')
    ax2 = axes[1, col]; ax2.set_facecolor('#1a1d27')
    ax2.bar(ks, sil, color=[color if k==bk else '#334455' for k in ks], edgecolor='#222')
    ax2.set_title(f'{label}\nSilhouette (best k={bk})', color='white', fontweight='bold')
    ax2.set_xlabel('k', color='#aaa'); ax2.set_ylabel('Silhouette', color='#aaa')
    ax2.tick_params(colors='#aaa'); ax2.set_xticks(ks)
    for s in ax2.spines.values(): s.set_color('#333344')
plt.tight_layout()
plt.savefig('../results/kmeans_elbow_silhouette.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
plt.show()

## 5. Business Insight

| Experiment | Features | Optimal k | What it reveals |
|---|---|---|---|
| Phase 1: RFM | 3 | ? | How customers *transact* |
| Phase 2: +Risk | 4 | ? | Who is financially *at-risk* |
| Phase 3: +Demo | 6 | ? | Full customer *identity* |

### 💡 The Story
> *When we look only at transaction behavior, we see N groups.*  
> *Adding credit risk reveals a hidden high-risk segment.*  
> *Adding demographics enables precision targeting.*  
> **All computation runs inside Teradata — zero data movement.**

In [ ]:
summary = []
for label, df, _ in experiments:
    v = df.dropna()
    best = v.loc[v.silhouette.idxmax()]
    summary.append({'Experiment': label, 'Best k': int(best.k), 'Silhouette': round(best.silhouette,4)})
pd.DataFrame(summary)

In [ ]:
remove_context()
print('Done ✅')